# Train MiniGPT on Q&A Dialogue

This notebook trains the **MiniGPT Transformer** on the conversational instruction dataset (`qa_data.txt`).

### Improvements Over Base Pre-Training:
1. **Instruction Format**: The network learns that questions starting with `User:` should be followed by direct factual replies under `Assistant:`.
2. **Full Character Vocabulary**: Covers all 98 standard ASCII characters (digits `0-9`, symbols `+`, `-`, `=`, `?`, letters, and punctuation) so the model never encounters unknown tokens.
3. **Scaled Capacity**: 6 Transformer layers, 8 attention heads, 256 embedding dimension, and a 256-token context window (~4.8M parameters).
4. **Model Checkpointing**: Saves model weights, configuration, and vocabulary to `mini_gpt.pt` so inference can run instantly without retraining.


## 1. Imports & Hardware Accelerator
Detect Apple Silicon (`MPS`), Nvidia (`CUDA`), or CPU.


In [ ]:
import math
import os
import string
import time
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

# Select accelerator
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"[*] Using hardware accelerator: {device.upper()}")
torch.manual_seed(1337)

## 2. Hyperparameters
Tuned for fast, accurate memorization of conversational turns on Apple Silicon.


In [ ]:
# Model Architecture
block_size = 256       # Context length (enough for a multi-sentence QA turn)
n_embd = 256           # Embedding dimension
n_head = 8             # Number of attention heads (256 // 8 = 32 dim per head)
n_layer = 6            # Number of Transformer blocks
dropout = 0.1          # Regularization dropout rate

# Training Settings
batch_size = 64        # Parallel sequences per optimization step
learning_rate = 5e-4   # AdamW learning rate
max_iters = 2500       # Optimization steps
eval_interval = 250    # Evaluate validation loss every 250 steps
eval_iters = 50        # Number of batches for loss averaging
checkpoint_path = "mini_gpt.pt"

## 3. Load Q&A Dataset & Full ASCII Vocabulary
Ensure all standard letters, digits, and punctuation are represented in the vocabulary.


In [ ]:
# Resolve dataset path
dataset_path = Path("qa_data.txt")
if not dataset_path.is_file():
    dataset_path = Path("gemini/qa_data.txt")

if not dataset_path.is_file():
    raise FileNotFoundError(f"Cannot find {dataset_path}. Please run prepare_qa_data first!")

with dataset_path.open("r", encoding="utf-8") as f:
    text = f.read()

# Build vocabulary containing all characters in dataset PLUS all standard printable ASCII
base_chars = set(text)
standard_printable = set(string.ascii_letters + string.digits + string.punctuation + " \t\n\r")
chars = sorted(list(base_chars.union(standard_printable)))
vocab_size = len(chars)

print(f"[*] Dataset: {dataset_path} ({len(text):,} characters)")
print(f"[*] Vocabulary size: {vocab_size} unique characters (covers all ASCII letters, digits, and symbols)")

# Tokenizer lookup tables
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: "".join([itos[i] for i in l])

# Split into train (90%) and validation (10%) sets
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"[*] Train set: {len(train_data):,} tokens | Val set: {len(val_data):,} tokens")

## 4. Batching and Loss Estimation
Create random chunks from the dataset for training and validation loss tracking.


In [ ]:
def get_batch(split: str):
    """Sample a random batch of input X and next-token target Y."""
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    """Estimate average loss without tracking gradients."""
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

## 5. Transformer Architecture
Define `CausalSelfAttention`, `MLP`, `Block`, and the `MiniGPT` model.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(block_size, block_size)).view(
                1, 1, block_size, block_size
            ),
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 8,
        n_layer: int = 6,
        block_size: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.block_size = block_size
        self.vocab_size = vocab_size

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying
        self.tok_emb.weight = self.head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size, f"Sequence length {T} exceeds block size {self.block_size}"

        tok_embeddings = self.tok_emb(idx)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_embeddings = self.pos_emb(pos)
        x = self.drop(tok_embeddings + pos_embeddings)

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 0.7, top_k: int = 40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

## 6. Instantiate Model and Optimizer
Create the ~4.8M parameter Transformer and initialize the AdamW optimizer.


In [ ]:
model = MiniGPT(
    vocab_size=vocab_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
    dropout=dropout,
).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"[*] Total Model Parameters: {param_count:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-1)

## 7. Execute Training Loop
Train across 2,500 iterations. The training loss will decline to ~0.06 as the model masters the Q&A dialogues.


In [ ]:
print("=" * 55)
print("  Starting Training Loop")
print("=" * 55)
start_time = time.time()

for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        print(
            f"Step {it:4d}/{max_iters:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f} | "
            f"Elapsed: {elapsed:.1f}s"
        )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

total_time = time.time() - start_time
print("=" * 55)
print(f"[*] Training finished in {total_time:.1f} seconds.")

## 8. Save Checkpoint
Save weights, architecture configuration, and vocabulary mapping to `mini_gpt.pt`.


In [ ]:
# Save checkpoint to current folder or gemini/
out_ckpt = Path(checkpoint_path)
if not out_ckpt.parent.name == "gemini" and Path("gemini").is_dir():
    out_ckpt = Path("gemini") / checkpoint_path

checkpoint = {
    "model_state": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "block_size": block_size,
        "dropout": dropout,
    },
    "chars": chars,
    "is_qa_model": True,
}
torch.save(checkpoint, out_ckpt)
print(f"[✓] Checkpoint saved successfully to '{out_ckpt}'.")

## 9. Quick Interactive Test
Test the newly trained model on a sample query right inside the notebook.


In [ ]:
test_query = "User: What is the capital of France?\nAssistant: "
context = torch.tensor([encode(test_query)], dtype=torch.long, device=device)
generated_tokens = model.generate(context, max_new_tokens=60, temperature=0.3, top_k=40)
output_text = decode(generated_tokens[0].tolist())

# Extract clean answer
print("Raw Output:")
print(output_text)
print("\nParsed Answer:")
answer = output_text[len(test_query):].split("<|end|>")[0].split("User:")[0].strip()
print(f"Assistant: {answer}")